# El model dels k veïns més propers (k-NN)

**Optativa d'Aprenentatge automàtic · DAM/DAW 2n**

Al quadern anterior heu vist que un ordinador pot trobar regles del tipus `if`/`elif`
soles, mirant dades. Ara toca entendre **un model de veritat**, des de la matemàtica
que hi ha a sota fins al codi que l'implementa.

Comencem pel més senzill de tots: el dels **k veïns més propers**, conegut com **k-NN**
(*k-nearest neighbors*). És el primer perquè la seva matemàtica és la que ja coneixeu:
el teorema de Pitàgoras.

## 1. El problema

Torneu a tenir el dataset de les flors d'iris: 150 flors, 3 espècies, 4 mesures per
flor (llargada i amplada del sèpal i del pètal, en centímetres).

Ara us donen una **flor nova**, sense etiquetar, amb les seves 4 mesures. No sabeu de
quina espècie és. Com ho decidiu?

La idea de k-NN es diu en una frase: **mira quines flors conegudes se li assemblen
més, i fes-les votar.** Si la majoria de les flors que més s'assemblen a la nova són
`versicolor`, dieu que la flor nova també ho és.

Tot el model es redueix a definir bé una paraula: **assemblar-se**.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
dades = iris.frame.copy()
dades["especie"] = iris.target_names[iris.target]
dades = dades.rename(columns={
    "sepal length (cm)": "sepal_llarg",
    "sepal width (cm)": "sepal_ample",
    "petal length (cm)": "petal_llarg",
    "petal width (cm)": "petal_ample",
})

cols = ["sepal_llarg", "sepal_ample", "petal_llarg", "petal_ample"]

print("Files:", len(dades))
dades.head()

## 2. La matemàtica: què vol dir «assemblar-se»

Assemblar-se és **estar a prop**. I «estar a prop» es pot mesurar: és una **distància**.

Si dues flors tenen mesures pràcticament iguals, estan a prop en l'espai de les seves
mesures. Si les mesures són molt diferents, estan lluny. k-NN no fa res més que
calcular distàncies i mirar quines són les més petites.

### La distància entre dos punts (això ja ho sabeu)

Agafeu dues flors i mireu només dues mesures seves: la llargada i l'amplada del pètal.
Cada flor és un punt en un pla, amb coordenades $(x, y)$.

La distància entre dos punts $(x_1, y_1)$ i $(x_2, y_2)$ és el **teorema de
Pitàgoras**: els catets són les diferències en cada eix, i la distància és la
hipotenusa.

$$d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$$

Ho fem amb dues flors reals del dataset: una `setosa` i una `versicolor`.

In [ ]:
import matplotlib.pyplot as plt

flor_a = dades.iloc[0]     # setosa
flor_b = dades.iloc[50]    # versicolor

x1, y1 = flor_a["petal_llarg"], flor_a["petal_ample"]
x2, y2 = flor_b["petal_llarg"], flor_b["petal_ample"]

plt.figure(figsize=(6, 5))
plt.scatter([x1], [y1], color="tab:blue", s=90, zorder=3,
            label=f"flor A ({flor_a['especie']})")
plt.scatter([x2], [y2], color="tab:orange", s=90, zorder=3,
            label=f"flor B ({flor_b['especie']})")

# els catets del triangle rectangle
plt.plot([x1, x2], [y1, y1], "k--", linewidth=1)   # catet horitzontal
plt.plot([x2, x2], [y1, y2], "k--", linewidth=1)   # catet vertical
plt.plot([x1, x2], [y1, y2], "r-", linewidth=2, label="distància (hipotenusa)")

plt.xlabel("Llargada del pètal (cm)")
plt.ylabel("Amplada del pètal (cm)")
plt.title("La distància entre dues flors és un triangle rectangle")
plt.legend()
plt.grid(alpha=0.3)
plt.axis("equal")
plt.show()

catet_x = abs(x1 - x2)
catet_y = abs(y1 - y2)
dist = (catet_x ** 2 + catet_y ** 2) ** 0.5
print(f"Catet horitzontal: {catet_x:.2f} cm")
print(f"Catet vertical:    {catet_y:.2f} cm")
print(f"Distància = √({catet_x:.2f}² + {catet_y:.2f}²) = {dist:.2f} cm")

Res de nou: és Pitàgoras amb noms diferents. La novetat ve ara.

### I amb 4 mesures, què fem?

Cada flor no té 2 mesures, en té 4: `sepal_llarg`, `sepal_ample`, `petal_llarg`,
`petal_ample`. Ja no es pot dibuixar un punt en 4 dimensions. Però la fórmula **és la
mateixa**, només amb més sumands:

$$d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2 + (z_1 - z_2)^2 + (w_1 - w_2)^2}$$

O escrit de manera general, per a $n$ mesures qualssevol:

$$d = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

**No cal poder imaginar-ho per poder-ho calcular.** No sabeu dibuixar un punt en 4
dimensions, però un ordinador no dibuixa res: resta, eleva al quadrat i suma. Li és
igual si són 4 columnes o 400.

Calculem la distància real entre les mateixes dues flors, ara amb les 4 mesures.

In [ ]:
import numpy as np

a = flor_a[cols].to_numpy(dtype=float)
b = flor_b[cols].to_numpy(dtype=float)

diferencies = a - b
dist_4d = np.sqrt(np.sum(diferencies ** 2))

print("Mesures flor A:", a)
print("Mesures flor B:", b)
print("Diferències:   ", diferencies)
print(f"Distància en 4 dimensions: {dist_4d:.2f}")

La distància amb només el pètal era `3.51`; amb les 4 mesures és `4.00`. Té sentit: hi
hem afegit dues diferències més (les del sèpal), i totes sumen a la mateixa arrel.

### No totes les distàncies són la mateixa

La que hem fet servir és la **distància euclidiana**: arrel de la suma de quadrats. És
la més habitual, però no l'única. Una alternativa és la **distància de Manhattan**: en
lloc d'elevar al quadrat i fer l'arrel, simplement se sumen les diferències en valor
absolut.

$$d_{\text{manhattan}} = \sum_{i=1}^{n} |a_i - b_i|$$

El nom ve dels carrers de Manhattan: si vas en quadrícula, no pots retallar en diagonal,
has de sumar els blocs que recorres en cada direcció. Sol donar una distància més gran
que l'euclidiana, i és menys sensible a una diferència molt gran en una sola columna.

In [ ]:
manhattan = abs(x1 - x2) + abs(y1 - y2)
euclidiana = ((x1 - x2) ** 2 + (y1 - y2) ** 2) ** 0.5

print(f"Distància euclidiana (pètal): {euclidiana:.2f} cm")
print(f"Distància de Manhattan (pètal): {manhattan:.2f} cm")

## 3. Implementa-ho a mà

Ara que la matemàtica és clara, k-NN complet són només dos passos:

1. Calcula la distància de la flor nova a **totes** les flors conegudes.
2. Queda't amb les $k$ més properes i fes-les **votar** per l'espècie més repetida.

Cap llibreria, només Python i numpy.

In [ ]:
def distancia(a, b):
    """Distància euclidiana entre dos punts (llistes o arrays de la mateixa mida)."""
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    return np.sqrt(np.sum((a - b) ** 2))


def classifica_knn(flor_nova, X, y, k):
    """Classifica flor_nova per votació dels k veïns més propers de (X, y)."""
    dists = [distancia(flor_nova, x) for x in X]
    idx_ordenat = np.argsort(dists)      # índexs de menor a major distància
    idx_veins = idx_ordenat[:k]          # els k més propers
    especies_veines = y[idx_veins]
    valors, comptes = np.unique(especies_veines, return_counts=True)
    return valors[np.argmax(comptes)]    # l'espècie més votada

Provem-ho amb una flor concreta. Agafem la flor de la fila 75, l'apartem de la resta
(fem com si no en sabéssim l'espècie) i demanem a la nostra funció que l'endevini
mirant-ne les 5 més properes.

In [ ]:
idx_prova = 75
flor_prova = dades.iloc[idx_prova]
resta = dades.drop(index=idx_prova)

X_resta = resta[cols].to_numpy(dtype=float)
y_resta = resta["especie"].to_numpy()

predit = classifica_knn(flor_prova[cols].to_numpy(dtype=float), X_resta, y_resta, k=5)

print("Espècie real:  ", flor_prova["especie"])
print("Espècie predita:", predit)

## 4. Comprova que la teva implementació coincideix amb scikit-learn

Una flor encertada no demostra res. Cal examinar-ho amb moltes flors que el model no
hagi vist mai, i per això es reparteixen les dades en dos grups abans d'entrenar:

- **Entrenament**: les flors que el model (o, en aquest cas, la nostra funció) pot
  mirar per decidir.
- **Examen**: les flors que es guarden apart i que ni la funció ni scikit-learn veuen
  fins al moment de comprovar els encerts.

Si les examinéssim amb flors que ja coneixen, no sabríem si han après o si s'ho han
memoritzat.

In [ ]:
from sklearn.model_selection import train_test_split

X = dades[cols].to_numpy(dtype=float)
y = dades["especie"].to_numpy()

X_entrena, X_examen, y_entrena, y_examen = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Per ensenyar: {len(X_entrena)} flors")
print(f"Per examinar: {len(X_examen)} flors")

Ara classifiquem **totes** les flors d'examen amb la nostra funció, amb $k=5$, i fem el
mateix amb `KNeighborsClassifier` de scikit-learn entrenat amb el mateix $k$ i el mateix
conjunt d'entrenament. Si l'has escrit bé, les prediccions han de coincidir.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

k = 5

# la nostra funció, flor a flor
prediccions_ma = np.array([
    classifica_knn(flor, X_entrena, y_entrena, k) for flor in X_examen
])

# scikit-learn, amb el mateix k i el mateix entrenament
model = KNeighborsClassifier(n_neighbors=k)
model.fit(X_entrena, y_entrena)
prediccions_sklearn = model.predict(X_examen)

coincidencia = np.mean(prediccions_ma == prediccions_sklearn)
precisio_ma = np.mean(prediccions_ma == y_examen)
precisio_sklearn = model.score(X_examen, y_examen)

print(f"Coincidència entre les dues implementacions: {coincidencia:.1%}")
print(f"Precisió de la nostra funció:    {precisio_ma:.1%}")
print(f"Precisió de scikit-learn:        {precisio_sklearn:.1%}")

**Aquest és el moment clau del quadern.** Les dues implementacions coincideixen en el
`100 %` de les 45 flors d'examen, i totes dues encerten el `97,8 %`.

`KNeighborsClassifier` no fa cap màgia que tu no hagis fet. Calcula distàncies com les
del punt 2 i vota com al punt 3. La llibreria només ho fa més ràpid i amb menys línies.
Si mai hi ha desacord entre una implementació a mà i scikit-learn, sol ser per com es
desempaten distàncies iguals, no perquè la idea sigui diferent.

## 5. Per què k importa

$k$ és quantes flors voten. Canviar-lo canvia el resultat, i no sempre en la direcció
que penses. Provem tots els valors de $k$ d'1 a 25 i mirem la precisió a l'examen.

In [ ]:
valors_k = range(1, 26)
precisions = []

for k in valors_k:
    model_k = KNeighborsClassifier(n_neighbors=k)
    model_k.fit(X_entrena, y_entrena)
    precisions.append(model_k.score(X_examen, y_examen))

plt.figure(figsize=(9, 5))
plt.plot(list(valors_k), precisions, marker="o")
plt.xlabel("k (nombre de veïns)")
plt.ylabel("Precisió a l'examen")
plt.title("Precisió segons k")
plt.grid(alpha=0.3)
plt.show()

millor_k = int(np.argmax(precisions)) + 1
print(f"k=1:  precisió {precisions[0]:.1%}")
print(f"k=25: precisió {precisions[-1]:.1%}")
print(f"Millor k: {millor_k}, amb precisió {max(precisions):.1%}")

El millor resultat surt amb `k=5` (`97,8 %`); en canvi `k=1` només arriba al `93,3 %`.
No és casualitat:

- **Amb `k=1`** la flor nova es classifica com la seva flor més propera, i prou. Si
  aquella flor concreta és una que està en una zona rara del gràfic (soroll, una
  mesura atípica), t'equivoques tu també. És un model que **memoritza** en lloc
  d'aprendre un patró general: molt sensible a cada punt individual.
- **Amb `k` molt gran** (mireu com cauen alguns valors cap al final del gràfic, per
  exemple `k=25` torna a baixar al `93,3 %`) fas votar tantes flors que acabes
  incloent-hi veïns d'altres espècies només perquè hi ha poques flors d'examen. El
  model **ho suavitza tot** i perd el detall de les fronteres entre espècies.

Amb el dataset iris, que és petit (només 45 flors d'examen), la corba no és una U neta:
puja i baixa perquè una sola flor mal classificada ja mou la precisió gairebé dos punts.
Amb un dataset gran, la tendència es veu més clara, però la lliçó és la mateixa: **`k`
petit = sobreajust (overfitting); `k` gran = infraajust**. Cal trobar-hi un equilibri,
i normalment es fa provant diversos valors, exactament com acabem de fer.

## 6. El parany de les escales

Aquí ve el punt que fa de k-NN un model especialment instructiu, perquè l'error és
fàcil de cometre i difícil de detectar si no en saps l'existència.

La distància és una resta i una suma de quadrats. **No sap què són centímetres ni
mil·límetres**: només veu números. Si una columna del teu dataset ve en una escala molt
més gran que les altres (per exemple, perquè algú l'ha mesurat en mil·límetres en lloc
de centímetres), aquella columna **dominarà** el càlcul de la distància, encara que no
sigui més important per decidir l'espècie.

Ho comprovem: multipliquem `sepal_ample` per 100 (com si l'haguéssim mesurat en una
unitat 100 cops més fina) i tornem a entrenar amb `k=5`.

In [ ]:
X_entrena_mal = X_entrena.copy()
X_examen_mal = X_examen.copy()

# sepal_ample és la columna d'índex 1
X_entrena_mal[:, 1] *= 100
X_examen_mal[:, 1] *= 100

model_mal = KNeighborsClassifier(n_neighbors=5)
model_mal.fit(X_entrena_mal, y_entrena)
precisio_mal = model_mal.score(X_examen_mal, y_examen)

print(f"Precisió abans (sense tocar res):        97,8 %")
print(f"Precisió amb sepal_ample x100:            {precisio_mal:.1%}")

La precisió cau de `97,8 %` a `66,7 %`, i no hem canviat cap idea del model: només les
unitats d'una columna. `sepal_ample` ara val entre 200 i 440 en lloc d'entre 2 i 4,4, i
com que la distància eleva al quadrat, aquesta única columna aixafa les altres tres.

La solució és **posar totes les columnes a la mateixa escala** abans d'entrenar.
`StandardScaler` de scikit-learn transforma cada columna perquè tingui mitjana 0 i
desviació típica 1: totes queden comparables, independentment de les unitats originals.

In [ ]:
from sklearn.preprocessing import StandardScaler

escalador = StandardScaler()
X_entrena_esc = escalador.fit_transform(X_entrena_mal)
X_examen_esc = escalador.transform(X_examen_mal)

model_esc = KNeighborsClassifier(n_neighbors=5)
model_esc.fit(X_entrena_esc, y_entrena)
precisio_esc = model_esc.score(X_examen_esc, y_examen)

print(f"Precisió amb sepal_ample x100 (sense escalar): 66,7 %")
print(f"Precisió amb StandardScaler:                    {precisio_esc:.1%}")

Amb `StandardScaler` la precisió puja de `66,7 %` a `91,1 %`. Fixeu-vos que no torna
exactament al `97,8 %` original: escalar no és desfer el x100, és posar **totes** les
columnes en peu d'igualtat, i això també canvia una mica el pes relatiu que tenien
abans les columnes que ja anaven bé. Tot i així, la lliça és clara: escalar repara quasi
tot el mal, i no escalar el pot causar.

**Regla pràctica per a la resta del curs**: amb models que es basen en distàncies (com
k-NN), si tens dubtes sobre si cal escalar, escala. El cost és una línia de codi; el
d'no fer-ho pot ser aquesta caiguda de 30 punts de precisió sense cap avís.

## 7. Pràctica

Els exercicis fan servir les variables ja carregades en aquest quadern
(`dades`, `X_entrena`, `X_examen`, `y_entrena`, `y_examen`, `cols`, `distancia`,
`classifica_knn`).

### Exercici 1 — Distància de Manhattan

Escriu una funció `distancia_manhattan(a, b)` que calculi la distància de Manhattan
(suma de valors absoluts de les diferències, com al punt 2). Fes una còpia de
`classifica_knn` que es digui `classifica_knn_manhattan` i que faci servir aquesta
distància en lloc de l'euclidiana. Classifica totes les flors d'examen amb `k=5` i
compara la precisió amb la de la distància euclidiana. Canvia gaire?

In [ ]:
# Exercici 1

def distancia_manhattan(a, b):
    # calcula aquí la distància de Manhattan
    pass


def classifica_knn_manhattan(flor_nova, X, y, k):
    # copia classifica_knn i canvia la distància que fa servir
    pass


# calcula aquí la precisió amb la distància de Manhattan i compara-la amb la de l'euclidiana

### Exercici 2 — Troba la k òptima amb un bucle

Al punt 5 hem mirat el gràfic a ull. Escriu un bucle que provi `k` d'1 a 30, es quedi
amb la precisió de cada valor i, al final, imprimeixi quin `k` ha donat la millor
precisió i quina és. No facis servir cap valor fix: que ho decideixi el bucle.

In [ ]:
# Exercici 2

millor_k_trobat = None
millor_precisio_trobada = 0

# recorre els valors de k, entrena un KNeighborsClassifier per cadascun
# i queda't amb el k que doni la precisió més alta

print(f"Millor k: {millor_k_trobat}, precisió: {millor_precisio_trobada:.1%}")

### Exercici 3 — El parany de les escales amb el dataset Wine

`load_wine` de scikit-learn té 178 vins descrits per 13 columnes (alcohol, magnesi,
intensitat de color, prolina...) en escales molt diferents entre elles —per exemple,
la prolina va de desenes a centenars, i el grau alcohòlic va d'11 a 15. És l'escenari
del punt 6, però encara més exagerat.

1. Carrega el dataset amb `from sklearn.datasets import load_wine`.
2. Separa'l en entrenament i examen (`test_size=0.3, random_state=42`).
3. Entrena un `KNeighborsClassifier` amb `k=5` **sense escalar** i anota la precisió.
4. Aplica `StandardScaler` i torna a entrenar amb el mateix `k`. Compara.

Abans d'executar-ho, fes una predicció: creus que la diferència serà més gran, més
petita o semblant a la d'iris? Per què?

In [ ]:
# Exercici 3
from sklearn.datasets import load_wine

# carrega el dataset, separa'l en entrenament/examen,
# entrena sense escalar i despres amb StandardScaler, i compara les precisions

### Exercici 4 — Quina distància fa servir scikit-learn per defecte?

`KNeighborsClassifier` té un paràmetre `metric` (per defecte `"minkowski"` amb `p=2`,
que és exactament la distància euclidiana). Crea dos models sobre les dades d'iris
(sense escalar), un amb `metric="euclidean"` i un amb `metric="manhattan"`, tots dos
amb `k=5`. Compara les seves precisions a l'examen amb les que tu vas calcular a l'
exercici 1.

In [ ]:
# Exercici 4

# crea els dos KNeighborsClassifier amb metric diferent, entrena'ls i compara precisions

## Resum

- k-NN es basa en una sola idea: **assemblar-se és estar a prop**, i «a prop» es
  mesura amb una distància.
- La **distància euclidiana** és el teorema de Pitàgoras amb tants sumands com
  columnes tinguis. No cal poder dibuixar-ho per poder-ho calcular.
- L'has implementat **tu mateix**, amb dues funcions, i has comprovat que coincideix
  amb `KNeighborsClassifier` de scikit-learn al `100 %` dels casos: la llibreria no fa
  res que tu no hagis fet, només ho fa més ràpid.
- El valor de **`k`** és un equilibri: massa petit memoritza soroll (sobreajust), massa
  gran ho suavitza tot (infraajust).
- **Les escales importen.** Una columna en unitats més grans que les altres pot
  dominar la distància sense que el model t'avisi. Escalar (`StandardScaler`) ho
  corregeix. Aquesta lliçó no és només de k-NN: la necessitareu a la resta del curs.

### I ara què

k-NN decideix mirant les dades més properes, cada vegada que classifica. El proper
model que veurem no mira exemples: aprèn una única equació general a partir de les
dades i després l'aplica a qualsevol punt nou sense tornar a mirar-les. És la
regressió logística.